## LAB_NO_10_22JZELE0480 

### University of Engineering and Technology Peshawar, Nowshera Campus

#### Course: Machine Learning Lab

#### Student Name: Ibraheem khan
#### Registration Number: 22jzele0480

# Lab 10: Multi-Layer Perceptron (MLP) for Time Series Forecasting
## Objective
Build and train a Multi-Layer Perceptron (MLP) model for univariate multi-step time series forecasting using the AEP hourly energy dataset.

## Table of Contents
1. [Imports](#imports)
2. [Model Architecture](#model)
3. [Callbacks & Training Setup](#callbacks)
4. [Data Loading & Preparation](#data)
5. [Model Training](#training)
6. [Evaluation](#evaluation)
7. [Fine Tuning](#finetune)
8. [Conclusion](#conclusion)

## 1. Imports <a id='imports'></a>

In [28]:
import os
os.chdir(r'C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10')

## 2. Model Architecture <a id='model'></a>

In [29]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
import os
from tensorflow.keras.callbacks import Callback

In [3]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [4]:
def MLP():
    model = Sequential()
    model.add(Flatten(input_shape=(time_steps , num_features)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    return model

In [5]:
model1 = MLP()
model1.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 504)               0         
                                                                 
 dense (Dense)               (None, 32)                16160     
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 16,193
Trainable params: 16,193
Non-trainable params: 0
_________________________________________________________________


In [6]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


## 3. Callbacks & Training Setup <a id='callbacks'></a>

In [7]:
checkpoints = r'C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [8]:
os.path.exists(JSON_PATH)

False

In [9]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor = TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1]

In [10]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =MLP()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [11]:
import os
path_dataset =r'C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

C:\Users\Ibraheem khan\anaconda3\envs\tf_env\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [12]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.008432626724243164 sec


In [13]:
train_X.shape

(835, 24, 21)

In [14]:
epochs = 23
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                    verbose = verbose)

Epoch 1/23
18/27 [===================>..........] - ETA: 0s - loss: 0.1779 - mae: 0.1779 - mape: 87.3500  
Epoch 1: val_loss improved from inf to 0.09905, saving model to C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-0001-loss0.10.h5
27/27 [==============================] - 1s 20ms/step - loss: 0.1499 - mae: 0.1499 - mape: 78.0865 - val_loss: 0.0991 - val_mae: 0.0991 - val_mape: 33.0609
Epoch 2/23
23/27 [========================>.....] - ETA: 0s - loss: 0.0742 - mae: 0.0742 - mape: 35.0822
Epoch 2: val_loss improved from 0.09905 to 0.05308, saving model to C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-0002-loss0.05.h5
27/27 [==============================] - 0s 5ms/step - loss: 0.0730 - mae: 0.0730 - mape: 36.5719 - val_loss: 0.0531 - val_mae: 0.0531 - val_mape: 16.7220
Epoch 3/23
22/27 [=======================>......] - ETA: 0s - loss: 0.0586 - mae: 0.0586 - mape: 28.7061
Epoch 3: val_loss improved from 0.05308 to 0.04090, saving model to C:\Users\Ibraheem khan\Downloads

In [16]:

model = load_model(r'C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-0008-loss0.03.h5',compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 146ms/step
Mean Absolute Error (MAE): 8982.98
Median Absolute Error (MedAE): 9624.29
Mean Squared Error (MSE): 81485558.45
Root Mean Squared Error (RMSE): 9026.94
Mean Absolute Percentage Error (MAPE): 57.47 %
Median Absolute Percentage Error (MDAPE): 62.19 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


# Fine Tuning

In [17]:
checkpoints = r'C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-0008-loss0.03.h5'
model=r'C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-0008-loss0.03.h5'
start_epoch= 24

In [19]:
from timeseires.callbacks.TrainingMonitor import TrainingMonitor

In [20]:
print(TrainingMonitor)

<class 'timeseires.callbacks.TrainingMonitor.TrainingMonitor'>


In [21]:
# 1
from timeseires.callbacks.TrainingMonitor import TrainingMonitor

# 2
TrainingMonitor1 = TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

In [22]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] loading C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-0008-loss0.03.h5...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


In [23]:
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
print(TrainingMonitor)
print(type(TrainingMonitor))

<class 'timeseires.callbacks.TrainingMonitor.TrainingMonitor'>
<class 'type'>


In [24]:
print(type(TrainingMonitor))

<class 'type'>


In [25]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
18/27 [===================>..........] - ETA: 0s - loss: 0.0335 - mae: 0.0335 - mape: 17.9685
Epoch 1: val_loss improved from inf to 0.03313, saving model to C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-0008-loss0.03.h5
27/27 [==============================] - 1s 14ms/step - loss: 0.0336 - mae: 0.0336 - mape: 16.8724 - val_loss: 0.0331 - val_mae: 0.0331 - val_mape: 10.4409
Epoch 2/10
21/27 [======================>.......] - ETA: 0s - loss: 0.0310 - mae: 0.0310 - mape: 16.3621
Epoch 2: val_loss did not improve from 0.03313
27/27 [==============================] - 2s 85ms/step - loss: 0.0313 - mae: 0.0313 - mape: 15.5317 - val_loss: 0.0371 - val_mae: 0.0371 - val_mape: 12.6129
Epoch 3/10
24/27 [=========================>....] - ETA: 0s - loss: 0.0302 - mae: 0.0302 - mape: 14.9678
Epoch 3: val_loss did not improve from 0.03313
27/27 [==============================] - 0s 14ms/step - loss: 0.0303 - mae: 0.0303 - mape: 15.0407 - val_loss: 0.0336 - val_mae: 0.0336 - val_map

## Evaluation <a id='evaluation'></a>

In [27]:

model = load_model(r'C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-0008-loss0.03.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 64ms/step
Mean Absolute Error (MAE): 8751.38
Median Absolute Error (MedAE): 9379.87
Mean Squared Error (MSE): 77569553.61
Root Mean Squared Error (RMSE): 8807.36
Mean Absolute Percentage Error (MAPE): 56.0 %
Median Absolute Percentage Error (MDAPE): 60.61 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)
